# Replaying an agent's trajectory: skrub, stratum, stratum's cache

An agent worked on the NYC housing task (which lots get a Class C HPD violation in
the next year) and wrote 32 pipelines, one after the other. Almost every pipeline
is an earlier one with one change: another table's features, another time window,
another model. The trajectory is `tests/applications/trajectories/nyc/pipelines/`, recorded in
stratum-bench (`trajectories/nyc-housing/mle-claude-run`).

This notebook replays the trajectory three times, each pipeline scored as the bench
scores it (a grid search with the task's expanding-window CV and average precision):

1. in plain skrub, in this kernel;
2. on the stratum server, without its reuse cache;
3. on the stratum server, with its cache, reusing eligible results from earlier
   pipelines.

The pipelines' code is not shown here; only what each one changed.

## 1. Start the server in a terminal

From the repository root, with the test and server dependencies installed (plus the `cpu` or `cu126`
extra for the neural-network pipeline):

```bash
python -m stratum serve --diagnostics-dir /tmp/stratum-runs
```

`--diagnostics-dir` is optional. With it, the server records its session and each
pipeline's run, and the inspector (`../stratum-gui`, started with
`STRATUM_DIAGNOSTICS_DIR=/tmp/stratum-runs`) shows them under **Server**.

## 2. Connect

`clear` empties the cache, so the cached replay starts from nothing, as the agent did.

In [ ]:
from stratum import StratumClient

client = StratumClient()  # the local server, found under ~/.stratum/server
client.clear()
client.health()

## 3. The lake

The agent read a lake of NYC open data on GCS: PLUTO, HPD violations, complaints and
litigations, DOB violations, 311 requests, ACRIS deeds and more. The replay reads a
generated lake with the same tables, columns and partitions, written next to this
notebook. `LOTS` and `EVENTS` set its size: the lots, and the rows of each event
table.

In [ ]:
import sys
from pathlib import Path

# examples/server/, whether the kernel starts there or in the project root.
HERE = next(d for d in (Path.cwd(), Path.cwd() / "examples" / "server")
            if (d / "nyc_trajectory.ipynb").exists())
ROOT = HERE.parents[1]
TRAJECTORY = ROOT / "tests" / "application" / "trajectories" / "nyc" / "pipelines"
sys.path[:0] = [str(ROOT), str(TRAJECTORY)]

from tests.applications.trajectories.nyc.test_nyc_trajectory import (
    PIPELINES, SCORING, _skip_reason, load_pipeline, make_lake)

LOTS, EVENTS = 3000, 300_000
LAKE = HERE / f"nyc_lake_{LOTS}_{EVENTS}"
if not LAKE.exists():
    make_lake(LAKE, n_lots=LOTS, n_events=EVENTS)
sorted(table.name for table in LAKE.iterdir())

## 4. The trajectory

Each pipeline names the pipeline it changes (`parent`) and says what it changed. A
pipeline that cannot run here (a missing optional dependency) is skipped.

In [ ]:
import ast

import pandas as pd

import common  # the trajectory's helpers, which read the lake from common.LAKE

common.LAKE = str(LAKE)


def header(name):
    # DESCRIPTION and PARENT, read without running the file.
    tree = ast.parse((TRAJECTORY / f"{name}.py").read_text())
    return {node.targets[0].id.lower(): ast.literal_eval(node.value) for node in tree.body
            if isinstance(node, ast.Assign) and isinstance(node.targets[0], ast.Name)
            and node.targets[0].id in ("DESCRIPTION", "PARENT")}


trajectory = {name: {**header(name), "skipped": _skip_reason(name)} for name in PIPELINES}
runnable = [name for name in PIPELINES if not trajectory[name]["skipped"]]

pd.set_option("display.max_colwidth", None)
pd.DataFrame(trajectory).T[["parent", "description", "skipped"]].fillna("")

## 5. Replaying

`replay` scores every pipeline in order with `search`, which takes a pipeline (built
lazily: the plan reads nothing until it runs) and returns its best score. It prints
one line per pipeline and returns the seconds and scores.

In [ ]:
import time


def replay(search):
    runs = {}
    for name in runnable:
        start = time.perf_counter()
        best = search(load_pipeline(name))
        runs[name] = {"seconds": time.perf_counter() - start, "score": best}
        print(f"{name}  {runs[name]['seconds']:6.1f}s  {SCORING} {best:.4f}")
    runs = pd.DataFrame(runs).T
    print(f"{runs['seconds'].sum():.0f}s in all")
    return runs

### 5.1 Plain skrub

In this kernel. Stratum is imported here, but leaves skrub's own `make_grid_search` in
charge while its scheduler is off.

In [ ]:
def skrub_search(pred):
    search = pred.skb.make_grid_search(fitted=True, refit=False, scoring=SCORING)
    return search.results_["mean_test_score"].max()


skrub = replay(skrub_search)

### 5.2 Stratum, without the cache

On the server. The client pickles the plan; the trajectory's helpers are not installed
on the server, so they travel with it, by value. `reuse_cache=False` runs the call
without the server's cache: it reads nothing from it and adds nothing.

In [ ]:
def stratum_search(pred, **config):
    results = client.grid_search(pred, scoring=SCORING, config={"implementation_selector": "default", **config})
    return results["scores"].max()


uncached = replay(lambda pred: stratum_search(pred, reuse_cache=False))

### 5.3 Stratum, with the cache

Replay the same calls with a shared cache. Later pipelines can reuse the lots,
labels, and shared violation and complaint features while entries remain cached.
New feature blocks and model configurations require additional computation.

The UDF that selects top-k categories is opaque. Its output and downstream
operations have no cache key, so those feature blocks are recomputed.

In [ ]:
client.clear()
before = client.stats()["stats"]
cached = replay(lambda pred: stratum_search(pred, reuse_cache=True))
after = client.stats()["stats"]
print(after["hits"] - before["hits"], "values from the cache,",
      after["puts"] - before["puts"], "added to it")

## 6. Compared

Check that each pipeline has the same score in all three replays, then compare
their execution times.

In [ ]:
scores = pd.DataFrame({"skrub": skrub["score"], "stratum": uncached["score"],
                       "stratum + cache": cached["score"]})
assert scores.nunique(axis=1).eq(1).all(), "the three replays score alike"

seconds = pd.DataFrame({"skrub": skrub["seconds"], "stratum": uncached["seconds"],
                        "stratum + cache": cached["seconds"]}).astype(float)
seconds.sum().round(0).to_frame("seconds in all").T

In [ ]:
import matplotlib.pyplot as plt

fig, (per_pipeline, total) = plt.subplots(2, 1, figsize=(12, 7))
seconds.plot.bar(ax=per_pipeline, width=0.8)
per_pipeline.set_ylabel("seconds")
per_pipeline.set_title("each pipeline")
seconds.cumsum().reset_index(drop=True).plot(ax=total, marker=".")
total.set_xticks(range(len(seconds)), seconds.index, rotation=90)
total.set_ylabel("seconds")
total.set_title("the trajectory so far")
fig.tight_layout()

The agent's progress: each pipeline's best score, and the best so far.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
score = scores["stratum + cache"].astype(float)
ax.plot(score.index, score, "o", label="pipeline")
ax.step(score.index, score.cummax(), where="post", label="best so far")
ax.set_ylabel(SCORING)
ax.tick_params(axis="x", rotation=90)
ax.legend()
fig.tight_layout()

## 7. In the inspector

With `--diagnostics-dir`, the **Server** view lists the session's grid_search calls,
two per pipeline (the uncached replay's and the cached one's), the cache's size over
time and the entries each call added; each call links its run, with the executed plan
and the reuse decision of each of its ops.

In [ ]:
{key: value for key, value in client.stats().items() if key != "stats"}

In [ ]:
client.close()